# Express Middleware

Middleware functions are code blocks that execute **sequentially** during the request–response lifecycle of an Express.js application. They sit between the incoming request from the client and the final route handler, with full access to:

- `req` — the request object
- `res` — the response object
- `next` — a function that hands control to the next middleware in the stack

Mental model: Express keeps an ordered array of functions. A request enters at index 0 and walks forward one function at a time. Each function either **ends** the trip (sends a response) or **forwards** it (`next()`). Nothing happens automatically — if neither occurs, the request just sits there.

```
Client Request
      ↓
[ express.json() ]      ← parses body
      ↓ next()
[ logger ]              ← logs method + url
      ↓ next()
[ checkAuth ]           ← may end the cycle here with 401
      ↓ next()
[ route handler ]       ← res.send() ends the cycle
      ↓
Client Response
```

---

## Core Mechanics

Every middleware function can do any of the following:

- Execute arbitrary code (calculations, DB queries, logging).
- Modify `req` and `res` (parse the body, attach `req.user`, set headers).
- **End** the request–response cycle (`res.send()`, `res.json()`, `res.render()`, `res.redirect()`).
- Pass control forward with `next()`.

> [!warning] The one rule that breaks everything
> If a middleware function does **not** end the request–response cycle, it **must** call `next()`. Otherwise the request hangs until the client times out. No error is thrown, nothing is logged — it just silently never responds. This is the single most common Express bug.

---

## Order of Execution Matters

Middleware runs in the order it is registered, top to bottom. This is not a detail — it is the whole model.

```javascript
app.use(express.json());       // 1. body is parsed...
app.post('/users', (req, res) => {
  console.log(req.body);       // 2. ...so req.body exists here
});
```

Flip those two and `req.body` is `undefined`. Same reason error handlers and 404 handlers must be registered **last**: anything registered after a matched route never runs.

---

## The 5 Types of Middleware

### 1. Application-Level

Bound to the `app` instance via `app.use()` or an HTTP method function like `app.get()`. Runs globally, or scoped to a path prefix.

```javascript
const express = require('express');
const app = express();

// Runs on every incoming request
app.use((req, res, next) => {
  console.log(`${req.method} request made to ${req.url}`);
  next();
});

// Runs only on requests starting with /admin
app.use('/admin', (req, res, next) => {
  console.log('Admin area accessed');
  next();
});
```

**Gotcha:** when you mount on a path prefix, that prefix is stripped from `req.url` inside the middleware. A request to `/admin/settings` sees `req.url === '/settings'`. Use `req.originalUrl` if you need the full path.

### 2. Router-Level

Identical behaviour, but bound to an `express.Router()` instance. This is how you isolate logic per module — scoping auth strictly to `/api/v1`, for example.

```javascript
const router = express.Router();

router.use((req, res, next) => {
  // Applies to every route on this router
  next();
});

router.get('/dashboard', (req, res) => {
  res.send('Dashboard');
});

module.exports = router;
```

```javascript
// In app.js
const apiRouter = require('./routes/api');
app.use('/api/v1', apiRouter);
```

### 3. Built-In

Ships with Express — no `npm install` needed. Since Express 4.16, the old `body-parser` package is bundled in, so `express.json()` and `express.urlencoded()` replace it.

| Middleware | Purpose |
|---|---|
| `express.json()` | Parses incoming JSON payloads into `req.body` |
| `express.urlencoded()` | Parses URL-encoded form data (HTML form posts) |
| `express.static()` | Serves static assets — images, CSS, client-side JS |

```javascript
app.use(express.json());
app.use(express.urlencoded({ extended: true }));
app.use(express.static('public'));
```

`extended: true` uses the `qs` library and allows nested objects in form data; `false` uses Node's `querystring` and only allows strings and arrays.

With `express.static('public')`, a file at `public/img/logo.png` is served at `/img/logo.png` — the directory name is not part of the URL.

### 4. Third-Party

Installed via npm. Common ones:

| Package | Purpose |
|---|---|
| `morgan` | Automated HTTP request logging |
| `cors` | Cross-Origin Resource Sharing headers |
| `helmet` | Sets a batch of security-related HTTP headers |
| `cookie-parser` | Populates `req.cookies` |
| `express-session` | Server-side session management |
| `method-override` | Fake PUT/DELETE from HTML forms via `?_method=DELETE` |
| `multer` | Handles `multipart/form-data` file uploads |

```javascript
const cors = require('cors');
const morgan = require('morgan');

app.use(cors());
app.use(morgan('dev'));
```

### 5. Error-Handling

These take **four** arguments: `(err, req, res, next)`. Express detects the arity and treats the function as an error handler, skipping it during normal flow and jumping to it when an error is thrown or passed via `next(err)`.

```javascript
app.use((err, req, res, next) => {
  console.error(err.stack);
  const { status = 500, message = 'Something went wrong' } = err;
  res.status(status).json({ error: message });
});
```

Three rules:

1. It must be registered **after** all routes and other middleware.
2. All four parameters must be declared, even if `next` is unused. Dropping it turns the function back into ordinary middleware.
3. If you call `next(err)` inside an error handler, control passes to the *next* error handler — useful for a logging handler followed by a responding handler.

---

## Route-Specific Middleware

Middleware doesn't have to be global. Pass it as an argument to a single endpoint, or as an array for several.

```javascript
const checkAuth = (req, res, next) => {
  if (!req.headers.authorization) {
    return res.status(401).send('Unauthorized Access'); // note the `return`
  }
  next();
};

app.get('/api/secure-data', checkAuth, (req, res) => {
  res.send('This is highly protected application data.');
});

// Multiple, in order
app.post('/api/posts', checkAuth, validateBody, (req, res) => {
  res.status(201).json({ created: true });
});
```

The `return` before `res.status(401).send()` is important. Without it, execution continues to `next()` and Express tries to send a second response, producing `Cannot set headers after they are sent to the client`.

---

## Passing Data Between Middleware

Attach to `req` for anything downstream middleware and handlers need:

```javascript
const loadUser = async (req, res, next) => {
  req.user = await User.findById(req.session.userId);
  next();
};
```

Use `res.locals` for data that templates should see — every view rendered on that request can read it without you passing it explicitly:

```javascript
app.use((req, res, next) => {
  res.locals.currentUser = req.user;
  next();
});
```

---

## `next()` vs `next('route')` vs `next(err)`

| Call | Effect |
|---|---|
| `next()` | Move to the next middleware in the stack |
| `next('route')` | Skip the remaining handlers **in this route** and move to the next matching route |
| `next(err)` | Skip all remaining normal middleware, jump straight to the error handlers |

```javascript
app.get('/user/:id',
  (req, res, next) => {
    if (req.params.id === '0') return next('route'); // bail to the next matching route
    next();
  },
  (req, res) => res.send('regular user')
);

app.get('/user/:id', (req, res) => res.send('special case'));
```

---

## The 404 Catch-All

Register a plain middleware after every route. If a request reaches it, nothing matched.

```javascript
// After all routes, before the error handler
app.use((req, res) => {
  res.status(404).json({ error: 'Not Found' });
});
```

---

## Async Errors: The Big Gotcha

In **Express 4**, a rejected promise inside an `async` handler is *not* caught. It becomes an unhandled rejection and the request hangs — your error handler never fires.

```javascript
// ❌ Express 4: error handler never runs
app.get('/posts', async (req, res) => {
  const posts = await Post.find({}); // if this throws, the request hangs
  res.json(posts);
});

// ✅ Express 4: catch and forward manually
app.get('/posts', async (req, res, next) => {
  try {
    res.json(await Post.find({}));
  } catch (err) {
    next(err);
  }
});
```

A wrapper avoids repeating that try/catch everywhere:

```javascript
const wrapAsync = (fn) => (req, res, next) =>
  Promise.resolve(fn(req, res, next)).catch(next);

app.get('/posts', wrapAsync(async (req, res) => {
  res.json(await Post.find({}));
}));
```

**Express 5** handles this natively — a returned rejected promise is forwarded to the error handler automatically, so `wrapAsync` becomes unnecessary. Check which major version the project is on before relying on it.

---

## Common Pitfalls

| Symptom | Cause |
|---|---|
| Request hangs forever | Forgot `next()` and didn't send a response |
| `req.body` is `undefined` | Missing `express.json()` / `express.urlencoded()`, or registered after the route |
| `Cannot set headers after they are sent` | Sent a response, then called `next()` or sent again — usually a missing `return` |
| Error handler never fires | Only 3 params declared, or it's registered before the routes |
| Async error swallowed | Express 4 without try/catch or a `wrapAsync` wrapper |
| Middleware never runs | Registered after a route that already matched and responded |

---

## Minimal Full Example

```javascript
const express = require('express');
const morgan = require('morgan');
const app = express();

// 1. Built-in + third-party
app.use(express.json());
app.use(express.urlencoded({ extended: true }));
app.use(express.static('public'));
app.use(morgan('dev'));

// 2. Custom
const checkAuth = (req, res, next) => {
  if (!req.headers.authorization) return res.status(401).send('Unauthorized');
  next();
};

// 3. Routes
app.get('/', (req, res) => res.send('Home'));
app.get('/secure', checkAuth, (req, res) => res.send('Protected'));

// 4. 404 — after all routes
app.use((req, res) => res.status(404).send('Not Found'));

// 5. Error handler — always last
app.use((err, req, res, next) => {
  console.error(err.stack);
  res.status(err.status || 500).send(err.message || 'Server Error');
});

app.listen(3000, () => console.log('Listening on 3000'));
```

---

## References

- [Express — Using middleware (official guide)](https://expressjs.com/en/5x/guide/using-middleware/)
- [Express — Error handling](https://expressjs.com/en/guide/error-handling.html)
- [GeeksforGeeks — Middleware in Express.js](https://www.geeksforgeeks.org/node-js/middleware-in-express-js/)
- [dev.to — What is middleware in Express and how it works](https://dev.to/satyasootar/what-is-middleware-in-express-and-how-it-works-47an)